# MolGap — SMILES Embedding Extraction

从 ChemBERTa 和 MolFormer 提取分子嵌入向量。

**使用前**：
1. Runtime → Change runtime type → **T4 GPU**
2. 左侧 📁 → 上传 `pubchemqc_chon_mw200_300_clean.csv`
3. 依次运行下面的单元格

In [ ]:
# Cell 1: 安装依赖
!pip install -q "transformers==4.40.2" "tokenizers<0.20" "huggingface_hub<0.25" torch pandas tqdm

In [ ]:
# Cell 2: 导入 & 检查 GPU
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 3: 加载数据
# 确保已通过左侧 📁 面板上传了 CSV 文件
INPUT_FILE = "pubchemqc_chon_mw200_300_clean.csv"

if not Path(INPUT_FILE).exists():
    raise FileNotFoundError(
        f"找不到 {INPUT_FILE}\n"
        "请先通过左侧文件面板上传"
    )

df = pd.read_csv(INPUT_FILE)
smiles_list = df["canonical_smiles"].tolist()
cid_list = df["cid"].tolist()
print(f"Loaded {len(smiles_list)} molecules")
df.head()

In [ ]:
# Cell 4: 嵌入提取函数
def extract_embeddings(model_name, smiles, cids, batch_size=64, max_length=128):
    from transformers import AutoTokenizer, AutoModel

    print(f"\n{'='*60}")
    print(f"Model: {model_name}")
    print(f"{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()

    dummy = tokenizer("C", return_tensors="pt", padding=True,
                      truncation=True, max_length=max_length).to(device)
    with torch.no_grad():
        dummy_out = model(**dummy)
    hidden_dim = dummy_out.last_hidden_state.shape[-1]
    print(f"Hidden dim: {hidden_dim}")

    all_embeddings = []
    n_batches = (len(smiles) + batch_size - 1) // batch_size

    for i in tqdm(range(n_batches), desc="Extracting"):
        batch = smiles[i * batch_size : (i + 1) * batch_size]
        tokens = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=max_length,
        ).to(device)

        with torch.no_grad():
            output = model(**tokens)

        hidden = output.last_hidden_state
        mask = tokens["attention_mask"].unsqueeze(-1)
        pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        all_embeddings.append(pooled.cpu().numpy())

    embeddings = np.vstack(all_embeddings)
    print(f"Embeddings shape: {embeddings.shape}")

    short_name = model_name.split("/")[-1].lower().replace("-", "_")[:12]
    col_names = [f"{short_name}_{j}" for j in range(embeddings.shape[1])]
    emb_df = pd.DataFrame(embeddings, columns=col_names)
    emb_df.insert(0, "cid", cids)
    return emb_df

In [ ]:
# Cell 5: 提取 ChemBERTa 嵌入
chemberta_df = extract_embeddings(
    model_name="seyonec/ChemBERTa-zinc-base-v1",
    smiles=smiles_list,
    cids=cid_list,
    batch_size=64,
)
chemberta_df.to_csv("embeddings_chemberta.csv", index=False)
print(f"Saved: embeddings_chemberta.csv {chemberta_df.shape}")

In [ ]:
# Cell 6: 提取 MolFormer 嵌入
molformer_df = extract_embeddings(
    model_name="ibm/MoLFormer-XL-both-10pct",
    smiles=smiles_list,
    cids=cid_list,
    batch_size=32,
    max_length=202,
)
molformer_df.to_csv("embeddings_molformer.csv", index=False)
print(f"Saved: embeddings_molformer.csv {molformer_df.shape}")

In [ ]:
# Cell 7: 合并 + 下载
merged = chemberta_df.merge(molformer_df, on="cid", how="inner")
merged.to_csv("embeddings_all.csv", index=False)
print(f"Merged: {merged.shape}")
print(f"  ChemBERTa: {chemberta_df.shape[1] - 1} dims")
print(f"  MolFormer: {molformer_df.shape[1] - 1} dims")
print(f"  Total: {merged.shape[1] - 1} dims")

from google.colab import files
files.download("embeddings_chemberta.csv")
files.download("embeddings_molformer.csv")
files.download("embeddings_all.csv")
print("\n下载完成！把 CSV 放到本地 data/processed/ 目录下。")